# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [2]:
!curl -L -o enron_spam_data.zip https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 27160  100 27160    0     0  23428      0  0:00:01  0:00:01 --:--:-- 23428


In [19]:
!rm -f enron_spam_data.zip
!curl -L -o enron_spam_data.zip https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 27160  100 27160    0     0  36057      0 --:--:-- --:--:-- --:--:-- 36057


In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

zsh:1: command not found: wget


In [25]:
!rm -f enron_spam_data.zip
!curl -L -o enron_spam_data.zip https://github.com/MWiechmann/enron_spam_data/archive/refs/heads/master.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 14.9M    0 14.9M    0     0  2860k      0 --:--:--  0:00:05 --:--:-- 5640k


In [26]:
import pandas as pd

In [27]:
# pandas can read the zip file directly
!unzip enron_spam_data.zip

Archive:  enron_spam_data.zip
e73c300263040f84b954e969ac06821c31cff47a
   creating: enron_spam_data-master
 extracting: enron_spam_data-master/.gitignore  
  inflating: enron_spam_data-master/LICENSE  
  inflating: enron_spam_data-master/README.md  
  inflating: enron_spam_data-master/build_data_file.py  
  inflating: enron_spam_data-master/enron_spam_data.zip  


In [24]:
!file enron_spam_data.zip

enron_spam_data.zip: HTML document text, Unicode text, UTF-8 text, with very long lines (11991)


In [28]:
import os
os.listdir()

['enron_spam_data.zip',
 'enron_spam_data-master',
 'project.ipynb',
 'queries.tsv',
 'README.md',
 '.git']

In [29]:
os.listdir("enron_spam_data-master")

['enron_spam_data.zip',
 'LICENSE',
 'README.md',
 'build_data_file.py',
 '.gitignore']

In [30]:
!unzip enron_spam_data-master/enron_spam_data.zip -d enron_spam_data-master

Archive:  enron_spam_data-master/enron_spam_data.zip
  inflating: enron_spam_data-master/enron_spam_data.csv  


In [31]:
import pandas as pd

df = pd.read_csv("enron_spam_data-master/enron_spam_data.csv")
df.head()

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [32]:
(df["Spam/Ham"] == "spam").mean()

0.5092834262664611

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [33]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

In [34]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [36]:
import requests
requests.get("https://huggingface.co").status_code

200

In [1]:
!curl -I https://huggingface.co

HTTP/1.0 307 Moved Temporarily
Server: Zscaler/6.2
Content-Length: 0
Location: ]8;;https://responsible-defenders-pages-production.s3.amazonaws.com/zscalerthree/GenAI_Block.html?url=https%3a%2f%2fhuggingface%2eco%2f&referer=&reason=Not+allowed+to+browse+Hugging+Face+category&reasoncode=CATEGORY_DENIED&timebound=1&action=deny&kind=category&rule=1075909&cat=Hugging+Face&user=violette.geoffrey@lmi.com&locid=00000000&lang=en_US&zsq=3j4TVP57rTQ1R642R4PJNnF64M1H5TJPs22sNqMzsq\https://responsible-defenders-pages-production.s3.amazonaws.com/zscalerthree/GenAI_Block.html?url=https%3a%2f%2fhuggingface%2eco%2f&referer=&reason=Not+allowed+to+browse+Hugging+Face+category&reasoncode=CATEGORY_DENIED&timebound=1&action=deny&kind=category&rule=1075909&cat=Hugging+Face&user=violette.geoffrey@lmi.com&locid=00000000&lang=en_US&zsq=3j4TVP57rTQ1R642R4PJNnF64M1H5TJPs22sNqMzsq
]8;;\


In [35]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

In [ ]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [ ]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [ ]:
# YOUR CHANGES HERE

...

Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Hint: Gradescope only allows files up to 100MB to be submitted.
Round your embeddings to 3 decimal places to make them smaller.

In [ ]:
# YOUR CHANGES HERE

...

Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [ ]:
# YOUR CHANGES HERE

...

Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [ ]:
# YOUR CHANGES HERE

...

Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [ ]:
# YOUR CHANGES HERE

...

Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [ ]:
# YOUR CHANGES HERE

...

Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.